Notebook 03 — Corrélations AT/MP × Expositions DARES======================================================Projet : Détection de biais dans les données de santé au travailAuteure : Laure Bonnefond — IPRP / IST — Master IA Data IPSSI BordeauxCe notebook croise les données de sinistralité AT/MP avec les expositionsprofessionnelles SUMER pour identifier les facteurs de risque les plusfortement corrélés aux accidents et maladies professionnelles.Méthodes :  - Corrélations de Pearson (linéaire)  - Corrélations de Spearman (monotone, rang)  - Tests de significativité (p-values)  - Scatter plots avec régression  - Détection des secteurs atypiques (sous/sur-déclaration)Objectif métier :  Identifier les expositions DARES qui prédisent le mieux la sinistralité,  et repérer les secteurs où la corrélation est faible (signal de  sous-déclaration potentielle).

# 1. Configuration


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.utils import PALETTE, CTN_LABELS

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Environnement prêt — Notebook 03 : Corrélations AT/MP × DARES")


# 2. Chargement et fusion des données


In [ ]:
df_atmp = pd.read_csv('../data/raw/atmp_par_secteur.csv')
df_sumer = pd.read_csv('../data/raw/sumer_expositions_secteur.csv')

# Fusion sur la clé CTN/secteur
df = df_atmp.merge(df_sumer, on=['ctn', 'secteur'])
print(f"Données fusionnées : {df.shape[0]} secteurs × {df.shape[1]} colonnes")

# Variables AT/MP (outcomes)
outcomes = {
    'indice_frequence': 'Indice de fréquence AT',
    'taux_gravite': 'Taux de gravité AT',
    'nb_at_graves': 'AT graves',
    'nb_mp': 'Maladies professionnelles',
}

# Variables d'exposition DARES (predictors)
expositions = [c for c in df.columns if c.startswith('pct_')
               and not any(x in c for x in ['hommes', 'femmes', '18_29',
                                              '30_44', '45_59', '60'])]
print(f"\nVariables AT/MP (outcomes)  : {len(outcomes)}")
print(f"Variables expositions DARES : {len(expositions)}")


# 3. Matrice de corrélation globale


## 3.1 Calcul des corrélations de Pearson


In [ ]:
# Construction de la matrice corrélation
matrice_corr = pd.DataFrame(
    index=[outcomes[o] for o in outcomes],
    columns=[e.replace('pct_', '').replace('_', ' ').title() for e in expositions]
)
matrice_pvalue = matrice_corr.copy()

for outcome_key, outcome_label in outcomes.items():
    for expo in expositions:
        expo_label = expo.replace('pct_', '').replace('_', ' ').title()
        r, p = stats.pearsonr(df[expo], df[outcome_key])
        matrice_corr.loc[outcome_label, expo_label] = round(r, 2)
        matrice_pvalue.loc[outcome_label, expo_label] = p

matrice_corr = matrice_corr.astype(float)
matrice_pvalue = matrice_pvalue.astype(float)

print("MATRICE DE CORRÉLATION (Pearson)")
print("=" * 70)
print(matrice_corr.to_string())


## 3.2 Heatmap des corrélations


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Annotations : valeur + étoiles de significativité
annot = matrice_corr.copy().astype(str)
for i in matrice_corr.index:
    for j in matrice_corr.columns:
        r = matrice_corr.loc[i, j]
        p = matrice_pvalue.loc[i, j]
        stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        annot.loc[i, j] = f'{r:.2f}{stars}'

sns.heatmap(matrice_corr, annot=annot, fmt='', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Coefficient de corrélation de Pearson'})
ax.set_title("Corrélations entre expositions DARES et sinistralité AT/MP\n"
             "(* p<0.05, ** p<0.01, *** p<0.001)")
ax.set_xlabel('Expositions professionnelles (SUMER)')
ax.set_ylabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../assets/03_heatmap_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/03_heatmap_correlations.png")


# 4. Top corrélations significatives


In [ ]:
print("TOP 15 CORRÉLATIONS LES PLUS FORTES (significatives p < 0.05)")
print("=" * 80)

resultats = []
for outcome_key, outcome_label in outcomes.items():
    for expo in expositions:
        expo_label = expo.replace('pct_', '').replace('_', ' ').title()
        r, p = stats.pearsonr(df[expo], df[outcome_key])
        if p < 0.05:
            resultats.append({
                'outcome': outcome_label,
                'exposition': expo_label,
                'r': round(r, 3),
                'p_value': round(p, 4),
                'direction': '↑' if r > 0 else '↓',
                'force': ('Forte' if abs(r) > 0.7
                          else 'Modérée' if abs(r) > 0.4
                          else 'Faible'),
            })

df_top = pd.DataFrame(resultats).sort_values('r', key=abs, ascending=False).head(15)
print(df_top.to_string(index=False))


# 5. Scatter plots des relations clés


In [ ]:
# Sélection des 4 corrélations les plus intéressantes pour scatter plots
relations_cles = [
    ('pct_contraintes_posturales', 'indice_frequence',
     'Contraintes posturales', 'Indice de fréquence AT'),
    ('pct_gestes_repetitifs', 'nb_mp',
     'Gestes répétitifs', 'Maladies professionnelles'),
    ('pct_travail_nuit', 'taux_gravite',
     'Travail de nuit', 'Taux de gravité'),
    ('pct_bruit_85db', 'nb_mp',
     'Exposition bruit > 85 dB', 'Maladies professionnelles'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, (expo, outcome, label_x, label_y) in enumerate(relations_cles):
    ax = axes[idx]
    x = df[expo].values
    y = df[outcome].values
    
    # Régression linéaire
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept
    
    # Scatter coloré par CTN
    for i, (xi, yi, ctn, sect) in enumerate(zip(x, y, df['ctn'], df['secteur'])):
        ax.scatter(xi, yi, color=PALETTE[i % len(PALETTE)],
                   s=180, alpha=0.8, edgecolors='white', linewidths=1.5,
                   label=f"CTN {ctn} — {sect[:25]}")
        ax.annotate(f'CTN {ctn}', (xi, yi), xytext=(5, 5),
                    textcoords='offset points', fontsize=9,
                    fontweight='bold')
    
    # Droite de régression
    ax.plot(x_line, y_line, '--', color='black', alpha=0.5, linewidth=1.5)
    
    # Annotation R² et p-value
    stars = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
    ax.text(0.05, 0.95,
            f'R = {r_value:.2f}{stars}\nR² = {r_value**2:.2f}\np = {p_value:.4f}',
            transform=ax.transAxes, fontsize=11,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    
    ax.set_xlabel(f'{label_x} (% salariés exposés)')
    ax.set_ylabel(label_y)
    ax.set_title(f'{label_x} vs {label_y}')

plt.suptitle('Relations clés entre expositions DARES et sinistralité AT/MP',
             fontsize=15, y=1.00)
plt.tight_layout()
plt.savefig('../assets/03_scatter_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/03_scatter_correlations.png")


# 6. Détection des secteurs atypiques

Pour chaque relation forte, on identifie les secteurs qui s'écartent
de la tendance générale. Ces écarts peuvent révéler :

- Une **sous-déclaration** (exposition forte mais sinistralité faible)
- Une **culture sécurité différenciée** (même exposition, moins d'AT)
- Une **sur-déclaration** (exposition faible mais sinistralité élevée)


In [ ]:
# Modèle de référence : indice fréquence = f(contraintes posturales)
x = df['pct_contraintes_posturales'].values
y = df['indice_frequence'].values
slope, intercept, _, _, _ = stats.linregress(x, y)
y_predit = slope * x + intercept
residus = y - y_predit
df['residu_at_postures'] = residus

print("ANALYSE DES ÉCARTS À LA TENDANCE")
print("=" * 75)
print("Outcome : Indice de fréquence AT vs Contraintes posturales")
print(f"Droite : IF = {slope:.2f} × %postures + {intercept:.1f}")
print("-" * 75)
print(f"{'Secteur':<35} {'Observé':>8} {'Prédit':>8} {'Résidu':>10} {'Lecture':<20}")
print("-" * 75)

for _, row in df.sort_values('residu_at_postures').iterrows():
    if row['residu_at_postures'] < -10:
        lecture = "📉 Sous-déclaration?"
    elif row['residu_at_postures'] > 10:
        lecture = "📈 Sur-sinistralité"
    else:
        lecture = "≈ Dans la tendance"
    print(f"{row['secteur'][:34]:<35} "
          f"{row['indice_frequence']:>8.1f} "
          f"{slope*row['pct_contraintes_posturales']+intercept:>8.1f} "
          f"{row['residu_at_postures']:>+10.1f} "
          f"{lecture}")


## 6.1 Visualisation des résidus


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

df_sorted = df.sort_values('residu_at_postures')
colors = ['#0F6E56' if r < -5 else '#E24B4A' if r > 5 else '#888780'
          for r in df_sorted['residu_at_postures']]

bars = ax.barh(df_sorted['secteur'], df_sorted['residu_at_postures'], color=colors)
ax.axvline(x=0, color='black', linewidth=1)
ax.set_xlabel("Écart à la tendance (résidu d'IF)")
ax.set_title(
    "Secteurs atypiques : écart entre AT observés et AT attendus\n"
    "(basé sur la corrélation contraintes posturales → indice de fréquence)"
)

# Légende et annotations
for bar, val, sect in zip(bars, df_sorted['residu_at_postures'], df_sorted['secteur']):
    label = f'{val:+.1f}'
    ax.text(val + (0.5 if val >= 0 else -0.5),
            bar.get_y() + bar.get_height()/2,
            label,
            va='center', ha='left' if val >= 0 else 'right', fontsize=10)

# Cadre interprétation
ax.text(0.99, 0.02,
        "📉 Négatif : moins d'AT que prévu → potentielle sous-déclaration\n"
        "📈 Positif : plus d'AT que prévu → sur-sinistralité",
        transform=ax.transAxes, fontsize=10,
        verticalalignment='bottom', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('../assets/03_residus_secteurs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/03_residus_secteurs.png")


# 7. Comparaison Pearson vs Spearman

Pearson mesure les relations linéaires. Spearman mesure les relations
monotones (non nécessairement linéaires) en utilisant les rangs.
Une grande différence Pearson/Spearman signale une relation non-linéaire
ou la présence d'outliers.


In [ ]:
print("COMPARAISON PEARSON vs SPEARMAN")
print("=" * 75)
print(f"{'Exposition':<30} {'Outcome':<20} {'Pearson':>10} {'Spearman':>10}")
print("-" * 75)

comparaisons = []
for expo in expositions:
    expo_label = expo.replace('pct_', '').replace('_', ' ').title()[:28]
    for outcome_key, outcome_label in list(outcomes.items())[:2]:
        r_pearson, _ = stats.pearsonr(df[expo], df[outcome_key])
        r_spearman, _ = stats.spearmanr(df[expo], df[outcome_key])
        diff = abs(r_pearson - r_spearman)
        flag = ' ⚠️ NL' if diff > 0.15 else ''
        comparaisons.append({
            'exposition': expo_label,
            'outcome': outcome_label,
            'pearson': round(r_pearson, 2),
            'spearman': round(r_spearman, 2),
            'diff': round(diff, 2),
        })
        if abs(r_pearson) > 0.4 or abs(r_spearman) > 0.4:
            print(f"{expo_label:<30} {outcome_label[:19]:<20} "
                  f"{r_pearson:>+10.2f} {r_spearman:>+10.2f}{flag}")

print("\n⚠️ NL = différence > 0.15 → relation potentiellement non-linéaire")


# 8. Synthèse visuelle pour le portfolio


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1 : Top corrélations avec IF
top_if = []
for expo in expositions:
    r, p = stats.pearsonr(df[expo], df['indice_frequence'])
    if p < 0.10:
        top_if.append({
            'expo': expo.replace('pct_', '').replace('_', ' ').title(),
            'r': r,
            'p': p,
        })

df_top_if = pd.DataFrame(top_if).sort_values('r', key=abs, ascending=True)
colors = ['#185FA5' if r > 0 else '#D85A30' for r in df_top_if['r']]
axes[0].barh(df_top_if['expo'], df_top_if['r'], color=colors)
axes[0].axvline(x=0, color='black', linewidth=1)
axes[0].set_xlabel('Corrélation avec indice de fréquence AT')
axes[0].set_title('Expositions prédictives des AT')
axes[0].set_xlim(-1, 1)
for i, (val, p) in enumerate(zip(df_top_if['r'], df_top_if['p'])):
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '·'
    axes[0].text(val + (0.02 if val >= 0 else -0.02), i,
                 f'{val:+.2f}{stars}',
                 va='center', ha='left' if val >= 0 else 'right',
                 fontsize=9)

# Panel 2 : Top corrélations avec MP
top_mp = []
for expo in expositions:
    r, p = stats.pearsonr(df[expo], df['nb_mp'])
    if p < 0.10:
        top_mp.append({
            'expo': expo.replace('pct_', '').replace('_', ' ').title(),
            'r': r,
            'p': p,
        })

df_top_mp = pd.DataFrame(top_mp).sort_values('r', key=abs, ascending=True)
colors = ['#185FA5' if r > 0 else '#D85A30' for r in df_top_mp['r']]
axes[1].barh(df_top_mp['expo'], df_top_mp['r'], color=colors)
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_xlabel('Corrélation avec nombre de MP')
axes[1].set_title('Expositions prédictives des MP')
axes[1].set_xlim(-1, 1)
for i, (val, p) in enumerate(zip(df_top_mp['r'], df_top_mp['p'])):
    stars = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '·'
    axes[1].text(val + (0.02 if val >= 0 else -0.02), i,
                 f'{val:+.2f}{stars}',
                 va='center', ha='left' if val >= 0 else 'right',
                 fontsize=9)

plt.suptitle('Quelles expositions DARES prédisent le mieux la sinistralité ?',
             fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../assets/03_top_predicteurs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/03_top_predicteurs.png")


# 9. Interprétation métier

## Ce que disent les corrélations

**Les expositions physiques prédisent bien les AT** :
- Contraintes posturales : très forte corrélation avec l'indice de fréquence
- Bruit > 85 dB et gestes répétitifs : prédicteurs forts des MP (TMS, surdité)
- Travail de nuit : associé à la gravité des AT (vigilance, fatigue)

**Les expositions tertiaires sont à l'opposé** :
- Travail sur écran > 6h/j : corrélé négativement avec les AT (logique :
  risque traumatique moindre dans les bureaux), mais ne capture pas
  les RPS qui sont moins déclarés en AT/MP

## Le signal le plus fort : les résidus

Les secteurs qui s'écartent fortement de la tendance sont les plus
intéressants pour la prévention :

- **Sous-déclaration possible** dans certains secteurs : exposition élevée
  mais peu d'AT déclarés. À investiguer : culture sécurité ? Pression sur
  les arrêts ? Effet "travailleur sain" ?

- **Sur-sinistralité** dans d'autres : moins d'exposition mais plus d'AT.
  Possibles facteurs : intérim, formation insuffisante, organisation
  défaillante.

## Limite méthodologique importante

Avec seulement 9 secteurs, la puissance statistique est limitée. Pour
une étude robuste, il faudrait descendre au niveau du **code NAF
détaillé** (700+ classes) ou exploiter les données régionales.

C'est exactement ce que ferait un projet en alternance avec accès aux
données détaillées.


# 10. Export des résultats


In [ ]:
# Sauvegarder les corrélations
matrice_corr.to_csv('../data/processed/03_matrice_correlations.csv')
matrice_pvalue.to_csv('../data/processed/03_pvalues_correlations.csv')
df.to_csv('../data/processed/03_atmp_sumer_fusionne.csv', index=False)

# Top corrélations triées
df_top.to_csv('../data/processed/03_top_correlations.csv', index=False)

print("Datasets exportés dans data/processed/ :")
print("  - 03_matrice_correlations.csv     (matrice 4×10)")
print("  - 03_pvalues_correlations.csv     (p-values)")
print("  - 03_atmp_sumer_fusionne.csv      (dataset complet)")
print("  - 03_top_correlations.csv         (top 15 significatives)")


In [ ]:
print("\n" + "=" * 60)
print("  NOTEBOOK 03 — CORRÉLATIONS TERMINÉES")
print("=" * 60)
print(f"\n  Corrélations testées          : {len(outcomes) * len(expositions)}")
print(f"  Significatives (p < 0.05)     : {len(df_top)}")
print(f"  Secteurs atypiques détectés   : "
      f"{(df['residu_at_postures'].abs() > 10).sum()}")
print(f"  Figures générées              : 4")
print(f"\n  Prochaine étape : 04_prediction_ml.ipynb")
print(f"  → Modèle Random Forest de prédiction du risque AT")
